For each NACE Class get the 100 chunks that scored highest across all the reports 

In [1]:
import pandas as pd
import glob
import os
import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [2]:
overview_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/PDF_stoxx600/STOXX600_as_of_2025_03_13.xlsx"

In [3]:
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_3_min_chunk_len_0_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/tables_cos_sim_0.0_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_2_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/sentence_len_5/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
raw_data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"

In [4]:
reports = glob.glob(raw_data_path + "*/*_long.csv")
reports = glob.glob(raw_data_path + "*/*_short.csv")

In [5]:
sample_ratio = 1

In [6]:
max_elements_per_class = 1000000

In [7]:
# if true, adds only chunks to training that have been classified into the same NACE class its report comes from
filter_only_right_chunks = True

In [8]:
# if true, adds random chunks that do not fulfill the minimum treshold (for BERT Training)
with_null_classifiers = False

In [9]:
new_threshold_cos_sin = 0.35

In [10]:
nace_level = 1

In [11]:
training_data_path = "../data/training_data/"

In [12]:
suffix = f"sample_ratio_{sample_ratio}" + ("__filter_only_right_chunks" if filter_only_right_chunks else "") + ("__with_null_classifiers" if with_null_classifiers else "")

# "_subsample" if sample_ratio != 1 else ""
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix)
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")
# end_path = os.path.join("/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications", raw_data_path.split("/")[-2] + suffix + "_with_no_class__cos_sim_04")

end_path = os.path.join(training_data_path, 
                        raw_data_path.split("/")[-2] + "__" + suffix)
end_path

'../data/training_data/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx__sample_ratio_1__filter_only_right_chunks'

In [13]:
df_overview = pd.read_excel(overview_path)
df_overview

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report
0,SalMar ASA,SALM-NO,SALM-NO,1984.965581,2458.824022,2271.611663,3.21,A,salmar-annual-report-2022.pdf
1,Bakkafrost P/F,BAKKA-NO,BAKKA-NO,929.503079,937.862170,973.053513,3.21,A,Bakkafrost PF2.pdf
2,Antofagasta plc,ANTO-GB,ANTO-GB,5577.681426,5849.975673,6113.946983,7.29,B,Antofagasta plc1.pdf
3,Anglo American plc,AAL-GB,AAL-GB,33423.271144,28355.894415,25288.188492,7.29,B,Anglo American plc1.pdf
4,TotalEnergies SE,TTE-FR,TTE-FR,250538.948328,202517.658053,180837.266896,6.10,B,Totalenergies EP Gabon1.pdf
...,...,...,...,...,...,...,...,...,...
595,Entain PLC,ENT-GB,ENT-GB,5036.123247,5484.020421,6011.923752,93.29,R,Entain PLC1.pdf
596,FDJ United,FDJU-FR,FDJU-FR,2461.100000,2621.400000,3065.000000,93.29,R,kindred-group_2022.pdf
597,Elis SA,ELIS-FR,ELIS-FR,3820.900000,4309.400000,4573.700000,96.01,S,Elis - 2022 financial statements.pdf
598,Scout24 SE,G24-DE,G24-DE,447.539000,509.114000,566.300000,96.09,S,Scout24 SE3.pdf


In [14]:
def get_all_level(nace_code, df_nace_codes_descriptions= None): 

    try:
        float(nace_code)
        if int(nace_code) < 10:
            nace_code = "0" + str(nace_code)

        init_level = len((str(nace_code)).replace(".",""))
    except ValueError:
        init_level = len((str(nace_code)).replace(".",""))

    levels = {init_level: nace_code}

    if df_nace_codes_descriptions is None: 
        df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")

    for i in range(0,init_level-1):         
        parent = df_nace_codes_descriptions[df_nace_codes_descriptions["ID"] == str(df_nace_codes_descriptions[df_nace_codes_descriptions["CODE"] == str(levels[init_level-i])]["PARENT_ID"].iloc[0])]
        levels[init_level-i-1] = parent["CODE"].iloc[0]

    return levels

In [15]:
df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")
filter_level_1_classes = "ABCDEFGHIJKLMNOPQRSTUVW"

In [16]:
result = pd.DataFrame(columns=["Sentences", "Score", "NACE_Code"])

# loop over all reports and get the all the sentences 
for report in tqdm.tqdm(reports):
    df = pd.read_csv(report)

    if filter_only_right_chunks: 
        report_name = os.path.basename(report).replace(".txt_short.csv", "") + ".pdf"
        report_code = df_overview[df_overview["Report"]==report_name]["NACE"].iloc[0]
        report_code = get_all_level(report_code)[nace_level]
        filter_column = list(filter(lambda x: "Scores_"+str(report_code) in x, df.columns))
        df = df[["Sentences"]+filter_column]

    if nace_level == 1: 
        scores = df[[column for column in df.columns if ("Scores" in column) and get_all_level(column.split("_")[1], df_nace_codes_descriptions)[nace_level] in filter_level_1_classes]].columns
    else: 
        scores = df[[column for column in df.columns if ("Scores" in column)]].columns
    
    for score in scores:
        temp = df[df[score].notna()][["Sentences", score]]  
        try: 
            temp["NACE_Code"] = get_all_level(score.split("_")[1])[nace_level]
        except IndexError: 
            continue
        temp = temp.rename(columns={score: "Score"})
        result = pd.concat([result, temp])

  0%|          | 0/256 [00:00<?, ?it/s]/var/folders/fp/yhl61lbj3m73x3_17tsp1vrr0000gn/T/ipykernel_18339/4209225735.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, temp])
100%|██████████| 256/256 [00:04<00:00, 63.66it/s]


In [17]:
result

,Sentences,Score,NACE_Code
2,where the provisions indicated by ceding compa...,0.405393,K
4,in preparing the annual financial statements t...,0.370080,K
9,term repurchase agreements repos were entered ...,0.387733,K
10,the technical interest results in the main fro...,0.394837,K
11,shares units or shares in investment funds bea...,0.414052,K
...,...,...,...
88,if you wish to participate via teleconference ...,0.134618,L
89,financial items net improved from negative sek...,0.143327,L
90,cash flow from financing activities totalled s...,0.251828,L
91,continued strong growth in net sales following...,0.209037,L


In [18]:
os.makedirs(end_path, exist_ok=True)

In [19]:
recordings = []

In [20]:
# for each code, store the 100 with the highest similarity score to the code

full_df = []
for code in set(result["NACE_Code"].to_list()): 

    #if not get_all_level(code.split("_")[1], df_nace_codes_descriptions)[1] in filter_level_1_classes: 
    # if not get_all_level(code, df_nace_codes_descriptions)[nace_level] in filter_level_1_classes: 
    #     continue

    temp = result[result["NACE_Code"] == code]
    temp = temp.drop_duplicates(subset="Sentences")
    temp = temp[temp["Sentences"].apply(len) >= 100]
    temp = temp.sort_values(by="Score", ascending=False)

    if with_null_classifiers: 
        temp.loc[temp["Score"]<new_threshold_cos_sin, "NACE_Code"] = "NO_CLASS"
        
        class_index = temp[temp["NACE_Code"]!="NO_CLASS"].index
        no_class_index = temp[temp["NACE_Code"]=="NO_CLASS"].index

        print(len(temp[temp["NACE_Code"]=="NO_CLASS"]))
        print(temp[temp["NACE_Code"]=="NO_CLASS"].index)
        print(temp.loc[no_class_index])
        print(code)
        print("--")

        temp = temp.loc[list(np.random.choice(no_class_index, len(class_index)))+list(class_index)]
    else:
        temp = temp[temp["Score"] >= new_threshold_cos_sin]

    number_of_elements_per_class = min(int(sample_ratio*len(temp)), max_elements_per_class, len(temp))
    random_choice = np.random.choice(len(temp), number_of_elements_per_class, replace=False)
    temp = temp.iloc[random_choice]
    temp = temp.reset_index(drop=True)
    temp["Evaluation"] = None
    temp["Notes"] = None
    temp = temp[["Evaluation", "Notes", "Sentences", "Score", "NACE_Code"]]

    recordings.append({"Code": code,"Nbr. of Chunks": len(temp),"Avg. Length": temp["Sentences"].apply(len).mean(), "Avg. Score": temp["Score"].mean()})

    text = ""
    for i, row in temp.iterrows():
        text += f"#{i}, Score: " + str(round(row["Score"], 2)) + "\n\n" + row["Sentences"] + "\n\n"

    with open(os.path.join(end_path, code.replace("/"," ")) + ".txt", "w") as f:
        f.write(text)

    temp.to_csv(os.path.join(end_path, code.replace("/"," ")) + ".csv")

    full_df.append(temp)

full_df = pd.concat(full_df, axis=0, ignore_index=True)

In [21]:
df_recordings = pd.DataFrame(recordings)
df_recordings.head()

,Code,Nbr. of Chunks,Avg. Length,Avg. Score
0,D,715,443.186014,0.469644
1,R,20,291.100000,0.398011
2,K,4370,361.221510,0.472197
3,E,500,400.870000,0.496045
4,S,76,345.947368,0.381869


In [22]:
df_recordings.to_csv(end_path + "/statistics.csv")

In [23]:
full_df= full_df.rename(columns={"Sentences": "text"})
#full_df = full_df.drop(columns="Score")
full_df

,Evaluation,Notes,text,Score,NACE_Code
0,None,None,it seeks to transform the means of transport o...,0.467702,D
1,None,None,enags has been carrying out the majority of it...,0.470606,D
2,None,None,sgn rotherhithe limited england and wales r pr...,0.501881,D
3,None,None,capital expenditure in dcc lpg primarily compr...,0.363762,D
4,None,None,electricity generators large electricity deman...,0.528642,D
...,...,...,...,...,...
12665,None,None,food hygiene and quality mean sufficient acces...,0.444077,I
12666,None,None,the companys compensation policy for its chair...,0.380085,I
12667,None,None,these components were determined by the board ...,0.381441,I
12668,None,None,people worldwide including those in our corpor...,0.520208,I


In [24]:
from sklearn.model_selection import train_test_split

# Split full_df into train (60%) and temp (40%)
train_df, temp_df = train_test_split(full_df, test_size=0.4, random_state=42)

# Split temp into test (20%) and validation (20%)
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Print the sizes of each split
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}, Validation size: {len(val_df)}")

Train size: 7602, Test size: 2534, Validation size: 2534


In [25]:
full_df.to_csv(end_path + "/full_data.csv", index=False)

In [26]:
train_df.to_csv(end_path + "/train_data.csv", index=False)
val_df.to_csv(end_path + "/val_data.csv", index=False)
test_df.to_csv(end_path + "/test_data.csv", index=False)

In [27]:
end_path

'../data/training_data/paragraph_and_sentence_len_6_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx__sample_ratio_1__filter_only_right_chunks'